# einops for Deep Learning — worked examples (with plain torch/numpy equivalents)

Based on the official tutorial: https://einops.rocks/2-einops-for-deep-learning/

Part 1 introduced `rearrange` / `reduce` / `repeat` on images. This part shows how the
*same three verbs* express the tensor gymnastics that show up everywhere in deep-learning
models — layout conversions, flatten-before-FC, pooling, global average pooling,
normalization, channel shuffle, anchor/bbox splitting — and how `einops.layers` lets you
drop these patterns straight into an `nn.Sequential`.

Two ideas to keep in mind:

- **Framework-agnostic.** einops dispatches on the tensor type. The *exact same* pattern
  string runs on numpy, PyTorch, TensorFlow, JAX, etc. Here we use PyTorch.
- **Differentiable.** `rearrange`/`reduce` are ordinary ops in the autograd graph —
  gradients flow through them.

Convention throughout: tensors are **channel-first** `b c h w` (PyTorch's NCHW), with
`x` of shape `(10, 32, 100, 200)`. For each example we run einops, then reproduce it with
raw `reshape`/`permute`/`mean`/... and assert the two match exactly — so you can see what
einops expands to.

In [2]:
import numpy as np
import torch
from einops import rearrange, reduce, repeat, asnumpy, parse_shape
import einops
print("einops", einops.__version__, "| torch", torch.__version__)

# A batch of 10 feature maps, 32 channels, 100x200 spatial. float64 for exact checks.
x = torch.from_numpy(np.random.RandomState(42).normal(size=[10, 32, 100, 200]))
print("x.shape =", tuple(x.shape))


def to_np(t):
    if isinstance(t, torch.Tensor):
        return t.detach().cpu().numpy()
    return np.asarray(t)


def check(einops_out, plain_out, tol=1e-9):
    a, b = to_np(einops_out), to_np(plain_out)
    assert a.shape == b.shape, f"shape mismatch: {a.shape} vs {b.shape}"
    assert np.allclose(a, b, atol=tol), "VALUES DIFFER"
    print(f"einops == reference  ✓   shape = {a.shape}")

einops 0.8.2 | torch 2.11.0+cu130
x.shape = (10, 32, 100, 200)


## Part 1 — Simple computations & layout conversions

The single most common einops call in a DL codebase: convert between **channel-first**
(`NCHW`, PyTorch/conv default) and **channel-last** (`NHWC`, TensorFlow / image libs).

### 1. Channel-first → channel-last

`"b c h w -> b h w c"` is a pure permutation; the numpy/torch form is a `permute`.

In [3]:
e = rearrange(x, "b c h w -> b h w c")
p = x.permute(0, 2, 3, 1)
print("shape:", tuple(e.shape))   # (10, 100, 200, 32)
check(e, p)

shape: (10, 100, 200, 32)
einops == reference  ✓   shape = (10, 100, 200, 32)


### 2. Backpropagation through einops

einops ops are part of the autograd graph. Below, `reduce(..., 'max')` then `sum` builds a
scalar loss; `.backward()` populates `x.grad`. A `max` reduction routes the gradient to
the arg-max position of each `(b, c)` plane, so exactly `b * c = 320` entries get a `1`
and the total gradient sums to 320.

In [5]:
xg = x.clone().requires_grad_(True)

y1 = reduce(xg, "b c h w -> b c", "max")   # max over each (h, w) plane
y2 = rearrange(y1, "b c -> c b")           # transpose (no effect on the sum)
loss = reduce(y2, "c b -> ", "sum")        # reduce ALL axes -> scalar
loss.backward()

total_grad = reduce(xg.grad, "b c h w -> ", "sum")
print("sum of grad =", float(total_grad), "  (== b*c =", 10 * 32, ")")
assert float(total_grad) == 10 * 32

sum of grad = 320.0   (== b*c = 320 )


### 3. `einops.asnumpy`

A backend-agnostic "give me a numpy array" — it detaches and moves off-GPU for you,
regardless of whether the input is a torch / TF / JAX tensor.

In [6]:
arr = asnumpy(y1)
print(type(arr), arr.shape)
assert isinstance(arr, np.ndarray)

<class 'numpy.ndarray'> (10, 32)


## Part 2 — Building blocks: flatten, space↔depth

Composition `(a b)` and decomposition `(a b) -> a b` express the reshapes that glue a
conv stack to a dense head, and the pixel-shuffle tricks used for up/down-sampling.

### 4. Flatten for a fully-connected layer

`"b c h w -> b (c h w)"` keeps the batch axis and merges everything else into one feature
vector — exactly what you do before the first `nn.Linear`. = `x.reshape(b, -1)`.

In [7]:
e = rearrange(x, "b c h w -> b (c h w)")
p = x.reshape(10, -1)
print("shape:", tuple(e.shape))   # (10, 640000)
check(e, p)

shape: (10, 640000)
einops == reference  ✓   shape = (10, 640000)


### 5. Space-to-depth

`"b c (h h1) (w w1) -> b (h1 w1 c) h w"` with `h1=w1=2` packs each 2×2 spatial block into
the channel axis: spatial shrinks by 2×, channels grow 4×. The numpy form is
reshape → permute → reshape.

In [8]:
e = rearrange(x, "b c (h h1) (w w1) -> b (h1 w1 c) h w", h1=2, w1=2)
# (b, c, h, h1, w, w1) -> want (b, h1, w1, c, h, w) -> merge (h1 w1 c)
p = (x.reshape(10, 32, 50, 2, 100, 2)
       .permute(0, 3, 5, 1, 2, 4)
       .reshape(10, 2 * 2 * 32, 50, 100))
print("shape:", tuple(e.shape))   # (10, 128, 50, 100)
check(e, p)

shape: (10, 128, 50, 100)
einops == reference  ✓   shape = (10, 128, 50, 100)


### 6. Depth-to-space (the inverse)

`"b (h1 w1 c) h w -> b c (h h1) (w w1)"` unpacks channels back into 2×2 spatial blocks:
channels shrink 4×, spatial grows 2×. This is the einops view of PyTorch's `PixelShuffle`.

In [9]:
e = rearrange(x, "b (h1 w1 c) h w -> b c (h h1) (w w1)", h1=2, w1=2)
# channels 32 -> (h1=2, w1=2, c=8); (b, h1, w1, c, h, w) -> (b, c, h, h1, w, w1)
p = (x.reshape(10, 2, 2, 8, 100, 200)
       .permute(0, 3, 4, 1, 5, 2)
       .reshape(10, 8, 200, 400))
print("shape:", tuple(e.shape))   # (10, 8, 200, 400)
check(e, p)

shape: (10, 8, 200, 400)
einops == reference  ✓   shape = (10, 8, 200, 400)


## Part 3 — Reductions: pooling & global pooling

Any axis on the left but not the right is reduced. Decomposing first turns reductions into
pooling of any window size, in any number of dimensions, with one consistent notation.

### 7. Global average pooling

`"b c h w -> b c"` with `"mean"` collapses both spatial axes — the standard head before a
classifier. = `x.mean(dim=(2, 3))`.

In [ ]:
e = reduce(x, "b c h w -> b c", "mean")
p = x.mean(dim=(2, 3))
print("shape:", tuple(e.shape))   # (10, 32)
check(e, p)

### 8. 2×2 max-pooling

`"b c (h h1) (w w1) -> b c h w"` with `"max", h1=2, w1=2`. einops also accepts the
shorthand `"b c (h 2) (w 2) -> b c h w"` (literal window size, no named axis).

In [ ]:
e = reduce(x, "b c (h h1) (w w1) -> b c h w", "max", h1=2, w1=2)
p = x.reshape(10, 32, 50, 2, 100, 2).amax(dim=(3, 5))
print("shape:", tuple(e.shape))   # (10, 32, 50, 100)
check(e, p)

# literal-window shorthand produces the identical result
e2 = reduce(x, "b c (h 2) (w 2) -> b c h w", "max")
check(e2, e)

### 9. Same notation, any dimensionality

1D pooling over a sequence and 3D pooling over a volume are the *same idea*; only the
number of decomposed axes changes.

In [ ]:
# 1D max-pool over time: (t b c) with windows of 2 along t
seq = torch.from_numpy(np.random.RandomState(0).normal(size=[8, 4, 16]))  # (t, b, c)
e = reduce(seq, "(t t1) b c -> t b c", "max", t1=2)
p = seq.reshape(4, 2, 4, 16).amax(dim=1)
print("1D pooled:", tuple(e.shape))   # (4, 4, 16)
check(e, p)

# 3D max-pool over a volume: (b c X Y Z) with 2x2x2 windows
vol = torch.from_numpy(np.random.RandomState(1).normal(size=[2, 3, 8, 8, 8]))
e = reduce(vol, "b c (X 2) (Y 2) (Z 2) -> b c X Y Z", "max")
p = vol.reshape(2, 3, 4, 2, 4, 2, 4, 2).amax(dim=(3, 5, 7))
print("3D pooled:", tuple(e.shape))   # (2, 3, 4, 4, 4)
check(e, p)

### 10. keepdims-style reductions for normalization

Writing `1` (or `()`) on the right keeps the axis at length 1 so the result broadcasts
back against `x`. `"b c h w -> b c 1 1"` is per-image-per-channel mean (instance-norm
style); `"b c h w -> 1 c 1 1"` is per-channel over the whole batch (batch-norm style).

In [ ]:
# per-image, per-channel
y = x - reduce(x, "b c h w -> b c 1 1", "mean")
p = x - x.mean(dim=(2, 3), keepdim=True)
print("shape:", tuple(y.shape))   # (10, 32, 100, 200)
check(y, p)

# per-channel across the whole batch
y = x - reduce(x, "b c h w -> 1 c 1 1", "mean")
p = x - x.mean(dim=(0, 2, 3), keepdim=True)
check(y, p)

### 11. Squeeze / unsqueeze for single-image inference

A literal `1` (or `()`) inserts or drops a unit axis — e.g. adding a batch axis of 1 in
front of a single image, then dropping a trailing classes-of-1 axis.

In [ ]:
image = rearrange(x[0, :3], "c h w -> h w c")   # one HWC image, 3 channels
batched = rearrange(image, "h w c -> 1 c h w")  # add a batch axis of 1
print("batched:", tuple(batched.shape))         # (1, 3, 100, 200)
check(batched, image.permute(2, 0, 1)[None])

logits = torch.from_numpy(np.random.RandomState(7).normal(size=[1, 10]))
squeezed = rearrange(logits, "1 classes -> classes")
print("squeezed:", tuple(squeezed.shape))        # (10,)
check(squeezed, logits.squeeze(0))

## Part 4 — Stacking, concatenation, shuffling, splitting

A Python **list of tensors** is treated as a new leading axis, so `np.stack`/`cat` become
patterns. The same composition machinery also expresses channel-shuffle and the
split-an-axis-into-named-parts idiom from detection heads.

### 12. Stack & concatenate from a list

`rearrange(list, ...)` introduces the list as the first axis `b`. Where you place `b`
decides stack-vs-concatenate and along which dimension.

In [ ]:
lst = list(x)   # 10 tensors, each (32, 100, 200)

# stack -> new axis, then go channel-last
e = rearrange(lst, "b c h w -> b h w c")
check(e, torch.stack(lst, 0).permute(0, 2, 3, 1))

# concatenate along height: merge b into h
e = rearrange(lst, "b c h w -> (b h) w c")
check(e, torch.stack(lst, 0).permute(0, 2, 3, 1).reshape(10 * 100, 200, 32))
print("concat-on-height:", tuple(e.shape))   # (1000, 200, 32)

# concatenate along channels: merge b into c
e = rearrange(lst, "b c h w -> h w (b c)")
check(e, torch.stack(lst, 0).permute(2, 3, 0, 1).reshape(100, 200, 10 * 32))
print("concat-on-channel:", tuple(e.shape))   # (100, 200, 320)

### 13. Channel shuffle (ShuffleNet)

`"b (g c) h w -> b (c g) h w"` with `g=4` regroups channels so information mixes across
groups after a grouped convolution. Same shape out, different ordering.

In [ ]:
e = rearrange(x, "b (g c) h w -> b (c g) h w", g=4)
# (b, g=4, c=8, h, w) -> swap g and c -> merge (c g)
p = x.reshape(10, 4, 8, 100, 200).permute(0, 2, 1, 3, 4).reshape(10, 32, 100, 200)
print("shape:", tuple(e.shape))   # (10, 32, 100, 200)
check(e, p)

### 14. Split an axis into named parts (detection head)

`"b (coord bbox) h w -> coord b bbox h w"` pulls the packed `coord` dimension to the front
so you can unpack it into named tensors, then compute on them.

In [ ]:
bbox_x, bbox_y, bbox_w, bbox_h = rearrange(
    x, "b (coord bbox) h w -> coord b bbox h w", coord=4, bbox=8
)
print("each part:", tuple(bbox_x.shape))   # (10, 8, 100, 200)

# reference: split channel into (coord=4, bbox=8), bring coord to front, index it
ref = x.reshape(10, 4, 8, 100, 200).permute(1, 0, 2, 3, 4)
check(bbox_x, ref[0]); check(bbox_h, ref[3])

max_bbox_area = reduce(bbox_w * bbox_h, "b bbox h w -> b h w", "max")
print("max bbox area:", tuple(max_bbox_area.shape))   # (10, 100, 200)

### 15. Packing order matters: consecutive vs strided split

`(split c)` puts `split` as the **outer** (slower) index → a *consecutive* (contiguous-half)
split. `(c split)` puts `split` **inner** → a *strided* (every-other) split. These are
genuinely different tensors — mixing them up silently corrupts e.g. bidirectional-LSTM
or GLU channels.

In [ ]:
# consecutive: first half of channels, then second half
c1, c2 = rearrange(x, "b (split c) h w -> split b c h w", split=2)
check(c1, x[:, :16]); check(c2, x[:, 16:])

# strided: even-indexed channels, then odd-indexed
s1, s2 = rearrange(x, "b (c split) h w -> split b c h w", split=2)
check(s1, x[:, 0::2]); check(s2, x[:, 1::2])

print("consecutive == strided?", torch.allclose(c1, s1))   # False

## Part 5 — `parse_shape`, striding-anything, and `einops.layers`

### 16. `parse_shape` — read named dimensions back out

Capture axis lengths into a dict so you can restore a shape later without hard-coding
numbers. `_` skips an axis.

In [ ]:
x5d = rearrange(x, "b c h (w z) -> b c h w z", z=20)
print("full:", parse_shape(x5d, "b c h w z"))
print("partial:", parse_shape(x5d, "batch c _ _ _"))
assert parse_shape(x5d, "b c h w z") == {"b": 10, "c": 32, "h": 100, "w": 10, "z": 20}

### 17. Striding anything

Fold a 2×2 stride into the batch axis, run a (dummy) per-pixel op, then fold it back —
turning any operation into a strided variant. The round-trip is exact.

In [ ]:
strided = rearrange(x, "b c (h hs) (w ws) -> (hs ws b) c h w", hs=2, ws=2)
print("strided batch:", tuple(strided.shape))   # (40, 32, 50, 100)

processed = strided * 1.0   # stand-in for some 2D op that preserves shape
back = rearrange(processed, "(hs ws b) c h w -> b c (h hs) (w ws)", hs=2, ws=2)
print("restored:", tuple(back.shape))           # (10, 32, 100, 200)
check(back, x)

### 18. `einops.layers` — patterns as `nn.Module`s

`Rearrange` and `Reduce` from `einops.layers.torch` are real layers: stateless, scriptable,
and droppable into `nn.Sequential`. A single `Reduce('b c (h 2) (w 2) -> b (c h w)', 'max')`
fuses the final pool **and** the flatten that usually precede a classifier head — no
hand-computed `view(x.size(0), -1)`.

In [ ]:
from torch.nn import Sequential, Conv2d, MaxPool2d, Linear, ReLU
from einops.layers.torch import Rearrange, Reduce

# A LeNet-ish stack on 32x32x3 inputs; note: no manual flatten anywhere.
model = Sequential(
    Conv2d(3, 6, kernel_size=5),    # 32 -> 28
    MaxPool2d(kernel_size=2),       # 28 -> 14
    Conv2d(6, 16, kernel_size=5),   # 14 -> 10
    Reduce("b c (h 2) (w 2) -> b (c h w)", "max"),   # pool 10->5 AND flatten
    Linear(16 * 5 * 5, 120),
    ReLU(),
    Linear(120, 10),
)

dummy = torch.randn(4, 3, 32, 32)
out = model(dummy)
print("model output:", tuple(out.shape))   # (4, 10)
assert tuple(out.shape) == (4, 10)

# A standalone Rearrange layer, e.g. patchify-then-project for a ViT stem:
patchify = Rearrange("b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1=4, p2=4)
print("patch tokens:", tuple(patchify(dummy).shape))   # (4, 64, 48)

## Takeaways

- The **same** `rearrange`/`reduce` patterns run unchanged on numpy, PyTorch, TF, JAX, and
  flow through autograd — `einops.asnumpy` gets you back to numpy from any backend.
- Layout conversions (NCHW↔NHWC), flatten-before-FC, space↔depth, pooling at any window
  size in 1D/2D/3D, global pooling, and keepdims-normalization are all one readable line.
- **Order inside a group encodes intent:** `(split c)` is a consecutive split, `(c split)`
  is strided — same shape, different data. Naming the axes makes that visible (and bugs
  catchable).
- `einops.layers.torch.{Rearrange, Reduce}` put these patterns directly into `nn.Sequential`,
  often replacing a manual `view(...).flatten()` (and the off-by-one shape bugs that come
  with it).